In [1]:
!pip install transformers datasets scikit-learn --quiet

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [2]:
train_df = pd.read_csv("train (3).csv")
val_df   = pd.read_csv("validation.csv")
test_df  = pd.read_csv("test (3).csv")

# Strip any extra spaces from column names
train_df.columns = train_df.columns.str.strip()
val_df.columns   = val_df.columns.str.strip()
test_df.columns  = test_df.columns.str.strip()

# Check columns
print("Train columns:", train_df.columns.tolist())


Train columns: ['sentence', 'label']


In [3]:
# Updated Class Count and Column Names
TEXT_COL = "sentence"  # Matches the column in your CSV
LABEL_COL = "label"    # The actual label column
N_CLASSES = 3          # negative (0), neutral (1), positive (2)

def prepare_data(df):
    # Select only the needed columns and rename for the model
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
    return df

# Apply the correction
train_df = prepare_data(pd.read_csv("train (3).csv"))
val_df   = prepare_data(pd.read_csv("validation.csv"))
test_df  = prepare_data(pd.read_csv("test (3).csv"))

In [4]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)


In [5]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

# Set torch format
train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
test_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/1449 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [6]:
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=N_CLASSES)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_steps=200,
    report_to="none"  # disables wandb
)


In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [10]:
trainer.train()


Step,Training Loss
50,0.985300
100,0.654800
150,0.498900
200,0.421300
250,0.470000
300,0.372500
350,0.352200


TrainOutput(global_step=364, training_loss=0.5268001949394142, metrics={'train_runtime': 143.3296, 'train_samples_per_second': 20.219, 'train_steps_per_second': 2.54, 'total_flos': 190625671143936.0, 'train_loss': 0.5268001949394142, 'epoch': 2.0})

In [11]:
trainer.evaluate()


{'eval_loss': 0.6315327882766724,
 'eval_accuracy': 0.8161157024793388,
 'eval_precision': 0.8397693458004867,
 'eval_recall': 0.8161157024793388,
 'eval_f1': 0.8188941462325363,
 'eval_runtime': 2.9461,
 'eval_samples_per_second': 164.287,
 'eval_steps_per_second': 20.706,
 'epoch': 2.0}